**生成式 AI 使用说明**：本作业中使用生成式 AI 工具时，适用与协作相同的课程政策。和其他协作者一样，每位学生都必须独立写出自己的解答，不能直接依赖交互输出；提交内容中还应注明协作的性质。使用生成式 AI 工具实质性完成作业内容不符合本作业的精神，也会违反 [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

**学生声明（必须填写）**

**SUNet id：**  
_填写你的 SUNet id_

**你是否使用了生成式 AI 工具（例如 ChatGPT、Copilot 等）？**  
- [ ] 否  
- [ ] 是（请在下方说明）

**如果使用了，请说明你如何使用生成式 AI，以及用于作业的哪些部分：**  
_填写你的回答_

**请确认所有提交内容都反映你自己的理解，并且你没有依赖 AI 完成作业的实质性部分：**  
- [ ] 我确认

In [ ]:
# 将你的 Google Drive 挂载到 Colab VM。
from google.colab import drive
drive.mount('/content/drive')

# TODO：填写 Drive 中保存解压后
# 作业文件夹，例如 'cs231n/assignments/assignment3/'
FOLDERNAME = 'cs231n/assignments/assignment3/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# 现在已经挂载 Drive，下面确保
# Colab VM 的 Python 解释器可以加载
# 其中的 Python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# 将 COCO 数据集下载到你的 Drive
# 如果它还不存在。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
# !bash get_数据集s.sh
!bash get_coco_dataset.sh
%cd /content/drive/My\ Drive/$FOLDERNAME


# 使用 Transformer 做图像描述
你已经为图像描述任务实现了 vanilla RNN。在这个 notebook 中，你将实现 transformer decoder 的关键部分，以完成同一任务。

**注意：** 与 RNN notebook 不同，本 notebook 主要使用 PyTorch，而不是 NumPy。

In [ ]:
# 设置单元。
import time, os, json
import numpy as np
import matplotlib.pyplot as plt
import sys
import types
import importlib

from cs231n.gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from cs231n.transformer_layers import *
from cs231n.captioning_solver_transformer import CaptioningSolverTransformer
from cs231n.classifiers.transformer import CaptioningTransformer
from cs231n.coco_utils import load_coco_data, sample_coco_minibatch, decode_captions
from cs231n.image_utils import image_from_url

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # 设置图像的默认大小。
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

if "imp" not in sys.modules:
    imp = types.ModuleType("imp")
    imp.reload = importlib.reload
    sys.modules["imp"] = imp

%load_ext autoreload
%autoreload 2

def rel_error(x, y):
    """ returns relative error """
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

# COCO 数据集
和前面的 notebook 一样，我们将使用 COCO 数据集做图像描述。

In [ ]:
# 从磁盘加载 COCO 数据到字典中。
data = load_coco_data(pca_features=True)

# 打印数据字典中的所有键和值。
for k, v in data.items():
    if type(v) == np.ndarray:
        print(k, type(v), v.shape, v.dtype)
    else:
        print(k, type(v), len(v))

# Transformer
你已经看到，RNN 非常强大，但通常训练较慢。此外，RNN 很难编码长距离依赖（LSTM 是缓解这个问题的一种方式）。2017 年，Vaswani 等人在论文 ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762) 中提出 Transformer，目的是引入并行性，并让模型学习长距离依赖。这篇论文不仅催生了自然语言处理领域中著名的 BERT 和 GPT 等模型，也引发了包括视觉在内的多个领域的广泛兴趣。这里我们在图像描述背景下介绍该模型，但 attention 思想本身更加通用。

# Transformer：Multi-Headed Attention

### Dot-Product Attention

回忆一下，attention 可以看作作用在一个 query $q\in\mathbb{R}^d$、一组 value vectors $\{v_1,\dots,v_n\}, v_i\in\mathbb{R}^d$，以及一组 key vectors $\{k_1,\dots,k_n\}, k_i \in \mathbb{R}^d$ 上的操作，定义为：

\begin{align}
c &= \sum_{i=1}^{n} v_i \alpha_i \\
\alpha_i &= \frac{\exp(k_i^\top q)}{\sum_{j=1}^{n} \exp(k_j^\top q)} \\
\end{align}

其中 $\alpha_i$ 通常称为 “attention weights”，输出 $c\in\mathbb{R}^d$ 是对 value vectors 的加权平均。

### Self-Attention
在 Transformer 中，我们执行 self-attention，也就是说 values、keys 和 query 都由输入 $X \in \mathbb{R}^{\ell \times d}$ 派生，其中 $\ell$ 是序列长度。具体来说，我们学习参数矩阵 $V,K,Q \in \mathbb{R}^{d\times d}$，把输入 $X$ 映射如下：

\begin{align}
v_i = Vx_i\ \ i \in \{1,\dots,\ell\}\\
k_i = Kx_i\ \ i \in \{1,\dots,\ell\}\\
q_i = Qx_i\ \ i \in \{1,\dots,\ell\}
\end{align}

### Multi-Headed Scaled Dot-Product Attention
在 multi-headed attention 中，我们为每个 head 学习一个参数矩阵，这让模型有更强表达能力，可以关注输入的不同部分。令 $h$ 为 head 数量，$Y_i$ 为第 $i$ 个 head 的 attention 输出。因此，我们学习单独的矩阵 $Q_i$、$K_i$ 和 $V_i$。为了让整体计算量与 single-headed 情况相同，我们选择 $Q_i \in \mathbb{R}^{d\times d/h}$、$K_i \in \mathbb{R}^{d\times d/h}$ 和 $V_i \in \mathbb{R}^{d\times d/h}$。在上面的简单 dot-product attention 中加入缩放项 $\frac{1}{\sqrt{d/h}}$ 后，有：

\begin{equation} \label{qkv_eqn}
Y_i = \text{softmax}\bigg(\frac{(XQ_i)(XK_i)^\top}{\sqrt{d/h}}\bigg)(XV_i)
\end{equation}

其中 $Y_i\in\mathbb{R}^{\ell \times d/h}$，$\ell$ 是序列长度。

在我们的实现中，会对 attention weights 应用 dropout（尽管实践中 dropout 也可以用于其他步骤）：

\begin{equation} \label{qkvdropout_eqn}
Y_i = \text{dropout}\bigg(\text{softmax}\bigg(\frac{(XQ_i)(XK_i)^\top}{\sqrt{d/h}}\bigg)\bigg)(XV_i)
\end{equation}

最后，self-attention 的输出是所有 head 拼接结果的线性变换：

\begin{equation}
Y = [Y_1;\dots;Y_h]A
\end{equation}

其中 $A \in\mathbb{R}^{d\times d}$，且 $[Y_1;\dots;Y_h]\in\mathbb{R}^{\ell \times d}$。

在 `cs231n/transformer_layers.py` 文件的 `MultiHeadAttention` 类中实现 multi-headed scaled dot-product attention。下面的代码会检查你的实现。相对误差应小于 `7e-3`。

In [ ]:
torch.manual_seed(231)

# Choose 维度 such 该 they are 所有 unique 用于 easier debugging:
# 具体来说, following 值 correspond 到 N=1, H=2, T=3, E//H=4, 并 E=8.
batch_size = 1
sequence_length = 3
embed_dim = 8
attn = MultiHeadAttention(embed_dim, num_heads=2)
attn.eval()

# Self-attention.
data = torch.randn(batch_size, sequence_length, embed_dim)
self_attn_output = attn(query=data, key=data, value=data)

# Masked self-attention.
mask = torch.randn(sequence_length, sequence_length) < 0.5
masked_self_attn_output = attn(query=data, key=data, value=data, attn_mask=mask)

# Attention 使用 two 输入.
other_data = torch.randn(batch_size, sequence_length, embed_dim)
attn_output = attn(query=data, key=other_data, value=other_data)

expected_self_attn_output = np.asarray([[
    [-0.1030,  0.2090,  0.7481, -0.4116, -0.2772,  0.2357, -0.2740, -0.2798],
    [-0.1794,  0.1637,  0.6930, -0.3435, -0.2300,  0.2457, -0.2504, -0.2323],
    [-0.1861,  0.1567,  0.6677, -0.3078, -0.2731,  0.2302, -0.2714, -0.2082]]])

expected_masked_self_attn_output = np.asarray([[
    [-0.1209,  0.1806,  0.8056, -0.4654, -0.2262,  0.2552, -0.2544, -0.2812],
    [-0.2374,  0.1183,  0.7423, -0.3777, -0.1460,  0.2716, -0.2213, -0.2077],
    [-0.0851,  0.2423,  0.7941, -0.3778, -0.4924,  0.1512, -0.4241, -0.1740]]])

expected_attn_output = np.asarray([[
    [-0.1965,  0.0702,  0.5227, -0.1388, -0.3530,  0.2163, -0.2643, -0.1902],
    [-0.3592, -0.0899,  0.5836,  0.0029, -0.3643,  0.2667, -0.1973, -0.1698],
    [-0.3795,  0.0433,  0.5634, -0.0749, -0.4379,  0.2714, -0.2207, -0.1980]]])

print('self_attn_output error: ', rel_error(expected_self_attn_output, self_attn_output.detach().numpy()))
print('masked_self_attn_output error: ', rel_error(expected_masked_self_attn_output, masked_self_attn_output.detach().numpy()))
print('attn_output error: ', rel_error(expected_attn_output, attn_output.detach().numpy()))

# Positional Encoding

虽然 transformer 可以轻松关注输入中的任意部分，但 attention 机制本身没有 token 顺序概念。然而，对许多任务（尤其是自然语言处理）来说，相对 token 顺序非常重要。为恢复顺序信息，作者向每个 word token 的 embedding 中加入 positional encoding。

令矩阵 $P \in \mathbb{R}^{l\times d}$，其中 $P_{ij} = $

$$
\begin{cases}
\text{sin}\left(i \cdot 10000^{-\frac{j}{d}}\right) & \text{if j is even} \\
\text{cos}\left(i \cdot 10000^{-\frac{(j-1)}{d}}\right) & \text{otherwise} \\
\end{cases}
$$

我们不会把输入 $X \in \mathbb{R}^{l\times d}$ 直接传入网络，而是传入 $X + P$。

在 `cs231n/transformer_layers.py` 中的 `PositionalEncoding` 里实现这一层。完成后，运行下面的代码做一个简单测试。你应该看到 `1e-3` 数量级或更小的误差。

In [ ]:
torch.manual_seed(231)

batch_size = 1
sequence_length = 2
embed_dim = 6
data = torch.randn(batch_size, sequence_length, embed_dim)

pos_encoder = PositionalEncoding(embed_dim)
pos_encoder.eval()
output = pos_encoder(data)

expected_pe_output = np.asarray([[[-1.1106,  1.0014,  1.5280, -0.0778, -0.6964,  1.1455],
                                  [ 0.8125, -0.4303,  0.4981,  0.7320,  1.1380,  1.5331]]])
print('pe_output error: ', rel_error(expected_pe_output, output.detach().numpy()))

# 内联问题 1

在设计上面介绍的 scaled dot product attention 时，有几个关键设计决策。解释为什么下列选择是有益的：
1. 使用多个 attention heads，而不是一个。
2. 在应用 softmax 函数之前除以 $\sqrt{d/h}$。回忆一下，$d$ 是特征维度，$h$ 是 head 数量。
3. 在 attention 操作输出后添加一个线性变换。

每个选择只需要一两句话，但请具体说明：如果没有该实现细节会发生什么，为什么这种情况不理想，以及所提出的实现如何改善它。

**你的回答：**

# Transformer Decoder Block

Transformer decoder layer 包含三个模块：（1）self attention，用于处理输入向量序列；（2）cross attention，用于基于可用上下文处理输入（在这里是图像特征）；（3）feedforward module，用于独立处理序列中的每个向量。完成 `cs231n/transformer_layers.py` 中 `TransformerDecoderLayer` 的实现，并在下面测试。相对误差应小于 `1e-6`。

Transformer decoder layer 的三个主要组件是：（1）处理输入向量序列的 self-attention 模块，（2）融合额外上下文的 cross-attention 模块（在这里是图像特征），以及（3）独立处理序列中每个向量的 feedforward 模块。完成 `cs231n/transformer_layers.py` 中 `TransformerDecoderLayer` 的实现，并在下面测试。相对误差应小于 `1e-6`。

In [ ]:
torch.manual_seed(231)
np.random.seed(231)

N, T, TM, D = 1, 4, 5, 12

decoder_layer = TransformerDecoderLayer(D, 2, 4*D)
decoder_layer.eval()
tgt = torch.randn(N, T, D)
memory = torch.randn(N, TM, D)
tgt_mask = torch.randn(T, T) < 0.5

output = decoder_layer(tgt, memory, tgt_mask)

expected_output = np.asarray([
    [[ 0.27005595, -0.21056601,  0.63376445, -0.35391954,  0.60071295,
      -1.7566615,  -0.1314159,  -1.7659538,  -0.5575517,   0.680143,
       1.8622686,   0.7291234],
     [-0.6080135,   0.4421802,  -0.07590749, -0.12093157,  0.94722354,
      -1.0223433,   0.47332686, -1.3958666,   2.1890075,  -1.0380646,
       0.9437816,  -0.7343922],
     [-0.89690155,  0.60447216,  0.10688882,  0.03492759,  1.8388084,
      -0.9772293,   0.7621334,  -0.668234,    1.3260399,  -1.8417019,
       0.21718435, -0.506388],
     [-0.70961505, -0.02918372, -0.39894593, -0.8069395,   0.04967685,
       0.29345992,  0.7871747,  -1.3172938,   1.4744806,   1.2633642,
       1.1611108,  -1.767289]]
])
print('error: ', rel_error(expected_output, output.detach().numpy()))

# 用 Transformer 做图像描述
现在你已经实现了前面的层，可以把它们组合成基于 Transformer 的图像描述模型。打开 `cs231n/classifiers/transformer.py` 文件，查看 `CaptioningTransformer` 类。

实现该类的 `forward` 函数。完成后，运行下面的代码，用一个小测试用例检查前向传播；你应该看到 `1e-5` 数量级或更小的误差。

In [ ]:
torch.manual_seed(231)
np.random.seed(231)

N, D, W = 4, 20, 30
word_to_idx = {'<NULL>': 0, 'cat': 2, 'dog': 3}
V = len(word_to_idx)
T = 3

transformer = CaptioningTransformer(
    word_to_idx,
    input_dim=D,
    wordvec_dim=W,
    num_heads=2,
    num_layers=2,
    max_length=30
)
transformer.eval()

features = torch.randn(N, D)
captions = torch.randint(0, V, (N, T))

scores = transformer(features, captions)

expected_scores = np.asarray([
    [[ 0.62165695, -0.6203744,  -1.116114  ],
     [ 0.6641097,  -0.60309553, -1.3479083 ],
     [ 0.62117404, -0.79205745, -1.1095243 ]],
    [[ 0.62160987, -0.6214724,  -1.1171186 ],
     [ 0.6093992,  -0.61166143, -1.3365762 ],
     [ 0.62108713, -0.79322916, -1.1104958 ]],
    [[ 0.6406587,  -0.57419956, -1.1300566 ],
     [ 0.6075228,  -0.60872054, -1.3372457 ],
     [ 0.6471472,  -0.8270494,  -1.1064028 ]],
    [[ 0.64029807, -0.5755552,  -1.1284486 ],
     [ 0.60708034, -0.6099963,  -1.3356377 ],
     [ 0.59367496, -0.8346836,  -1.0914496 ]]
])

print('scores error: ', rel_error(expected_scores, scores.detach().numpy()))

# 在小数据上过拟合 Transformer 图像描述模型
运行下面的代码，在与之前 RNN 相同的小数据集上，让基于 Transformer 的图像描述模型过拟合。

In [ ]:
torch.manual_seed(231)
np.random.seed(231)

data = load_coco_data(max_train=50)

transformer = CaptioningTransformer(
          word_to_idx=data['word_to_idx'],
          input_dim=data['train_features'].shape[1],
          wordvec_dim=256,
          num_heads=2,
          num_layers=2,
          max_length=30
        )


transformer_solver = CaptioningSolverTransformer(transformer, data, idx_to_word=data['idx_to_word'],
           num_epochs=100,
           batch_size=25,
           learning_rate=0.001,
           verbose=True, print_every=10,
         )

transformer_solver.train()

# Plot 训练 损失.
plt.plot(transformer_solver.loss_history)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Training loss history')
plt.show()

打印最终训练损失。你应该看到最终损失小于 0.05。

In [ ]:
print('Final loss: ', transformer_solver.loss_history[-1])

# 测试时的 Transformer 采样
采样代码已经为你写好。你只需运行下面的代码，把结果与之前的 RNN 结果进行比较。和之前一样，由于训练数据很少，训练集上的结果应该明显好于验证集结果。

In [ ]:
# If you get an error, URL just no longer exists, so don't worry!
# 你可以 re-sample as many times as you want.
for split in ['train', 'val']:
    minibatch = sample_coco_minibatch(data, split=split, batch_size=2)
    gt_captions, features, urls = minibatch
    gt_captions = decode_captions(gt_captions, data['idx_to_word'])

    sample_captions = transformer.sample(features, max_length=30)
    sample_captions = decode_captions(sample_captions, data['idx_to_word'])

    for gt_caption, sample_caption, url in zip(gt_captions, sample_captions, urls):
        img = image_from_url(url)
        # Skip missing URLs.
        if img is None: continue
        plt.imshow(img)
        plt.title('%s\n%s\nGT:%s' % (split, sample_caption, gt_caption))
        plt.axis('off')
        plt.show()

# Vision Transformer（ViT）

[Dosovitskiy et. al.](https://arxiv.org/abs/2010.11929) 表明，把 transformer 模型应用到图像 patch 序列上（称为 Vision Transformer）不仅能取得令人印象深刻的性能，而且在大数据集上训练时，相比卷积神经网络能更有效地扩展。我们将使用已有 transformer 组件实现一个 Vision Transformer 版本，并在 CIFAR-10 数据集上训练它。

Vision Transformer 会把输入图像转换为固定大小的 patch 序列，并把每个 patch embedding 成一个潜在向量。在 `cs231n/transformer_layers.py` 中完成 `PatchEmbedding` 的实现，并在下面测试。你应该看到小于 `1e-4` 的相对误差。

In [ ]:
from cs231n.transformer_layers import PatchEmbedding

torch.manual_seed(231)
np.random.seed(231)

N = 2
HW = 16
PS = 8
D = 8

patch_embedding = PatchEmbedding(
    img_size=HW,
    patch_size=PS,
    embed_dim=D
)

x = torch.randn(N, 3, HW, HW)
output = patch_embedding(x)

expected_output = np.asarray([
        [[-0.6312704 ,  0.02531429,  0.6112642 , -0.49089882,
          0.01412961, -0.6959372 , -0.32862484, -0.45402682],
        [ 0.18816411, -0.08142513, -0.9829535 , -0.23975623,
         -0.23109074,  0.97950286, -0.40997326,  0.7457837 ],
        [ 0.01810865,  0.15780598, -0.91804236,  0.36185235,
          0.8379501 ,  1.0191797 , -0.29667392,  0.20322265],
        [-0.18697818, -0.45137224, -0.40339014, -1.4381214 ,
         -0.43450755,  0.7651071 , -0.83683825, -0.16360264]],

       [[-0.39786366,  0.16201034, -0.19008337, -1.0602452 ,
         -0.28693503,  0.09791763,  0.26614824,  0.41781986],
        [ 0.35146567, -0.4469593 , -0.1841726 ,  0.45757473,
         -0.61304873, -0.29104248, -0.16124889, -0.14987172],
        [-0.2996967 ,  0.27353522, -0.09929767,  0.01973832,
         -1.2312065 , -0.6374332 , -0.22963578,  0.55696607],
        [-0.93818814,  0.02465284, -0.21117875,  1.1860403 ,
         -0.06137538, -0.21062079, -0.094347  ,  0.50032747]]])

print('error: ', rel_error(expected_output, output.detach().numpy()))

patch 向量序列会由 transformer encoder layers 处理，每层包含 self-attention 和 feed-forward 模块。由于所有向量都会相互 attend，attention masking 并非严格必要。不过，为保持一致性，我们仍然实现它。

在 `cs231n/transformer_layers.py` 中实现 `TransformerEncoderLayer`，并在下面测试。你应该看到小于 `1e-6` 的相对误差。

In [ ]:
torch.manual_seed(231)
np.random.seed(231)

from cs231n.transformer_layers import TransformerEncoderLayer

N, T, TM, D = 1, 4, 5, 12

encoder_layer = TransformerEncoderLayer(D, 2, 4*D)
encoder_layer.eval()
x = torch.randn(N, T, D)
x_mask = torch.randn(T, T) < 0.5

output = encoder_layer(x, x_mask)

expected_output = np.asarray([
    [[-0.48887438, -0.16470742,  0.7249629,  -1.0907345,   1.5764195,
       0.17484017,  0.8643941,  -0.5209561,  -1.6651545,  -1.1943513,
       1.4809005,   0.30326116],
     [-0.14663944,  1.052812,   -0.96227,    -0.20350897, -2.1478589,
       1.2637341,   0.04015125, -0.945255,    1.408903,   -0.43051198,
       0.37110543,  0.69933903],
     [ 0.08718029,  0.91050833, -1.208469,    1.1440394,  -1.9794976,
      -0.38342738, -0.47407156, -0.17345548,  0.11738186,  1.9534125,
       0.3525986,  -0.3461999],
     [-0.81389004,  0.76213264, -0.7094313,  -1.5332246,   1.3837544,
       0.4091982,   0.6569406,   0.7305365,  -1.197077,   -1.3698943,
       0.58292854,  1.0980265]]
])

print('error: ', rel_error(expected_output, output.detach().numpy()))

查看 `cs231n/classifiers/transformer.py` 中的 `VisionTransformer` 实现。

对于分类任务，ViT 会把输入图像划分为 patch，并用 transformer 处理 patch 向量序列。最后，对所有 patch 向量做 average pooling，并用于预测图像类别。我们会使用同样的一维 sinusoidal positional encoding 注入顺序信息，不过二维 sinusoidal positional encoding 和可学习 positional encoding 也都是有效选择。

完成 ViT 前向传播并在下面测试。你应该看到小于 `1e-6` 的相对误差。

In [ ]:
torch.manual_seed(231)
np.random.seed(231)
from cs231n.classifiers.transformer import VisionTransformer

imgs = torch.randn(3, 3, 32, 32)
transformer = VisionTransformer()
transformer.eval()

scores = transformer(imgs)

expected_scores = np.asarray(
    [[-0.15802915,  0.15734261, -0.06002094, -0.1860142,  -0.09376919,
      -0.08837911,  0.16224845,  0.0168601,   0.20606893,  0.13079852],
     [-0.12838936,  0.20798579, -0.0620573,  -0.11705838, -0.14088857,
      -0.11387511,  0.18839046,  0.03078492,  0.17386372,  0.06414902],
     [-0.09435399,  0.20495278, -0.03362986, -0.12232997, -0.15535004,
      -0.10380758,  0.21114832,  0.03481449,  0.18798208,  0.12231653]])

print('scores error: ', rel_error(expected_scores, scores.detach().numpy()))

我们首先通过让模型在一个训练 batch 上过拟合来验证实现。请相应调节 learning rate 和 weight decay。

In [ ]:
from torchvision import transforms
from torchvision.datasets import CIFAR10
from tqdm.auto import tqdm
from torch.utils.data import DataLoader

train_data = CIFAR10(root='data', train=True, transform=transforms.ToTensor(), download=True)
test_data = CIFAR10(root='data', train=False, transform=transforms.ToTensor(), download=True)

In [ ]:

learning_rate = 1e-4  # 可以尝试修改这里
weight_decay = 1.e-4  # 可以尝试修改这里


batch = next(iter(DataLoader(train_data, batch_size=64, shuffle=False)))
model = VisionTransformer(dropout=0.0)
loss_criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
model.train()

epochs = 100
for epoch in range(epochs):
    imgs, target = batch
    out = model(imgs)
    loss = loss_criterion(out, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    top1 = (out.argmax(-1) == target).float().mean().item()
    if epoch % 10 == 0:
      print(f"[{epoch}/{epochs}] Loss {loss.item():.6f}, Top-1 Accuracy: {top1:.3f}")


In [ ]:
# 你应该 get perfect 1.00 accuracy
print(f"Overfitting ViT on one batch. Top-1 accuracy: {top1}")

现在我们将在整个数据集上训练它。

In [ ]:
from cs231n.classification_solver_vit import ClassificationSolverViT

############################################################################
# TODO：训练 a Vision Transformer 模型 该 achieves 在 0.45 测试      #
# accuracy on CIFAR-10 after 2 epochs by adjusting 模型 architecture  #
# and/or 训练 参数 as 需要.                                    #
#                                                                          #
# 注意： If you want 到 使用 a GPU runtime, go 到 `Runtime > Change runtime  #
# type` 并 set `Hardware accelerator` 到 `GPU`. This 将 reset Colab,    #
# so 确保 到 rerun entire notebook 来自 beginning afterward.  #
############################################################################


learning_rate = 1e-4
weight_decay = 0.0
batch_size = 64
model = VisionTransformer()  # 你可以 want 到 change default params.




################################################################################
#                                 你的代码结束                             #
################################################################################

solver = ClassificationSolverViT(
    train_data=train_data,
    test_data=test_data,
    model=model,
    num_epochs = 2,  # Don't change 这个
    learning_rate = learning_rate,
    weight_decay = weight_decay,
    batch_size = batch_size,
)

solver.train('cuda' if torch.cuda.is_available() else 'cpu')



In [ ]:
print(f"Accuracy on test set: {solver.results['best_test_acc']}")

# 内联问题 2

尽管 ViT 最近在大规模图像识别任务中很成功，但在较小数据集上训练时，它们往往落后于传统 CNN。造成这种性能差距的底层因素是什么？可以使用哪些技术提升 ViT 在小数据集上的性能？

**你的回答**：在此填写。

# 内联问题 3

如果分别做下列改变，ViT 中 self-attention 层的计算成本会如何变化？请忽略 QKV 和输出投影的计算成本。

(i) hidden dimension 翻倍。
(ii) 输入图像的高度和宽度都翻倍。
(iii) patch size 翻倍。
(iv) layer 数量翻倍。

**你的回答**：在此填写。